<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 10 → Análise Preditiva e Prescritiva IoT
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

In [ ]:
# 1. PREVISÃO
np.random.seed(42)

tempo = pd.date_range("2026-09-29 14:00", periods=12, freq="1h")
h = np.arange(12)

previsao = (
    28
    + 4 * np.sin(np.pi * h / 11)
    + np.random.normal(0, 0.4, 12)
)

df = pd.DataFrame({
    "tempo": tempo,
    "temp_prevista": previsao
})

# 2. DECISÃO
def decidir(temp):
    if temp >= 31:
        return "Ventilar + Irrigar"
    if temp >= 29:
        return "Ventilar"
    return "Monitorar"

df["decisao"] = df["temp_prevista"].apply(decidir)

# 3. GRÁFICO - PREVISÃO
plt.figure(figsize=(10, 4))

plt.plot(df["tempo"], df["temp_prevista"], marker="o", label="Temperatura prevista")
plt.axhline(29, linestyle="--", label="Limite: ventilar")
plt.axhline(31, linestyle="--", label="Limite: ventilar + irrigar")

plt.title("PREVISÃO: Temperatura Futura")
plt.xlabel("Tempo")
plt.ylabel("Temperatura prevista (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. GRÁFICO - DECISÃO
mapa = {
    "Monitorar": 0,
    "Ventilar": 1,
    "Ventilar + Irrigar": 2
}

y = df["decisao"].map(mapa)

plt.figure(figsize=(10, 4))

plt.step(df["tempo"], y, where="mid")
plt.scatter(df["tempo"], y, s=60)

plt.yticks([0, 1, 2], ["Monitorar", "Ventilar", "Ventilar + Irrigar"])
plt.title("DECISÃO: Ação Recomendada")
plt.xlabel("Tempo")
plt.ylabel("Ação")
plt.grid(alpha=0.3)
plt.show()

# 5. RESULTADO
print(df.round({"temp_prevista": 1}))

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
# 1. DADOS
np.random.seed(42)
n = 100

df = pd.DataFrame({
    "tempo": pd.date_range("2026-09-29 08:00", periods=n, freq="10min"),
    "temperatura": 55 + 0.08*np.arange(n) + np.random.normal(0, 0.8, n),
    "vibracao": 2 + 0.025*np.arange(n) + np.random.normal(0, 0.15, n)
})

# 2. RISCO E STATUS
df["risco"] = (
    50 * ((df["temperatura"] - 55) / 10).clip(lower=0)
    + 50 * ((df["vibracao"] - 2) / 3).clip(lower=0)
).clip(0, 100)

df["status"] = np.select(
    [df["risco"] >= 70, df["risco"] >= 50],
    ["Manutenção", "Atenção"],
    default="Normal"
)

# 3. GRÁFICO DOS SENSORES
plt.figure(figsize=(11, 4))
plt.plot(df["tempo"], df["temperatura"], color="blue", label="Temperatura (°C)")
plt.plot(df["tempo"], df["vibracao"] * 12, color="orange", label="Vibração (escala visual)")
plt.title("Evolução dos Sensores da Máquina")
plt.xlabel("Tempo")
plt.ylabel("Nível do sinal")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 4. GRÁFICO DO RISCO
plt.figure(figsize=(11, 4))
plt.plot(df["tempo"], df["risco"], color="black", label="Risco")
plt.axhline(50, color="orange", linestyle="--", label="Atenção")
plt.axhline(70, color="red", linestyle="--", label="Manutenção")
plt.title("Índice de Risco para Manutenção Preditiva")
plt.xlabel("Tempo")
plt.ylabel("Risco (0-100)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 5. EXEMPLOS DOS STATUS
exemplos = pd.concat([
    df[df["status"] == "Normal"].tail(3),
    df[df["status"] == "Atenção"].head(5),
    df[df["status"] == "Manutenção"].head(5)
])

print(exemplos[["tempo", "temperatura", "vibracao", "risco", "status"]].round(2))

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
# 1. DADOS
np.random.seed(42)
n = 80
t = np.arange(n)

df = pd.DataFrame({
    "tempo": pd.date_range("2026-09-29 08:00", periods=n, freq="10min"),
    "temperatura": 55 + 0.09*t + np.random.normal(0, 0.7, n),
    "vibracao": 2 + 0.025*t + np.random.normal(0, 0.12, n)
})

# 2. SCORE, PROBABILIDADE E DECISÃO
df["score"] = (
    0.55 * (df["temperatura"] - 55) / 8
    + 0.45 * (df["vibracao"] - 2) / 2.5
)

df["probabilidade"] = 1 / (1 + np.exp(-5 * (df["score"] - 0.55)))

threshold = 0.70
df["decisao"] = np.where(df["probabilidade"] >= threshold, "Alerta", "Normal")

# 3. SCORE
plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["score"], color="blue", label="Score de risco")
plt.axhline(0.55, color="orange", linestyle="--", label="Referência do score")
plt.title("Score de Risco ao Longo do Tempo")
plt.xlabel("Tempo")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 4. PROBABILIDADE
plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["probabilidade"], color="green", label="Probabilidade de falha")
plt.axhline(threshold, color="red", linestyle="--", label="Threshold = 0,70")
plt.title("Probabilidade de Falha e Threshold")
plt.xlabel("Tempo")
plt.ylabel("Probabilidade")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 5. RESULTADOS
print(df.tail(12)[["tempo", "score", "probabilidade", "decisao"]].round(2))

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
# 1. DADOS
np.random.seed(42)
n = 200

prob = np.random.beta(2, 5, n)
falha_real = (np.random.rand(n) < prob).astype(int)

custo_fp, custo_fn = 200, 2000

# 2. TESTAR THRESHOLDS
resultados = []

for th in np.arange(0.1, 1.0, 0.1):
    alerta = (prob >= th).astype(int)

    fp = ((alerta == 1) & (falha_real == 0)).sum()
    fn = ((alerta == 0) & (falha_real == 1)).sum()
    custo = fp * custo_fp + fn * custo_fn

    resultados.append([th, fp, fn, custo])

res = pd.DataFrame(
    resultados,
    columns=["threshold", "falsos_positivos", "falsos_negativos", "custo_total"]
)

melhor = res.loc[res["custo_total"].idxmin()]

# 3. CUSTO
plt.figure(figsize=(9, 4))
plt.plot(res["threshold"], res["custo_total"], marker="o")
plt.axvline(
    melhor["threshold"],
    linestyle="--",
    label=f"Melhor threshold = {melhor['threshold']:.1f}"
)
plt.title("Custo Total da Decisão por Threshold")
plt.xlabel("Threshold")
plt.ylabel("Custo total (R$)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. ERROS
plt.figure(figsize=(9, 4))
plt.plot(res["threshold"], res["falsos_positivos"], marker="o", label="Falsos positivos")
plt.plot(res["threshold"], res["falsos_negativos"], marker="o", label="Falsos negativos")
plt.title("Erros de Decisão por Threshold")
plt.xlabel("Threshold")
plt.ylabel("Quantidade de erros")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 5. RESULTADOS
print(res)
print(f"\nMelhor threshold: {melhor['threshold']:.1f}")
print(f"Menor custo: R$ {melhor['custo_total']:.2f}")